# Word2Vec (Mikolov et al., 2013) — Implementation / 구현

**Paper**: Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). "Efficient Estimation of Word Representations in Vector Space." arXiv:1301.3781.

This notebook implements the two key architectures from the paper — **CBOW** and **Skip-gram** — from scratch using NumPy, then compares them with the modern `gensim` library.

이 노트북은 논문의 두 가지 핵심 아키텍처 — **CBOW**와 **Skip-gram** — 을 NumPy로 직접 구현하고, 현대적인 `gensim` 라이브러리와 비교합니다.

## Contents / 목차
1. **Part 1**: Vocabulary & Training Data / 어휘 및 학습 데이터 구축
2. **Part 2**: CBOW from Scratch / CBOW 직접 구현
3. **Part 3**: Skip-gram from Scratch / Skip-gram 직접 구현
4. **Part 4**: Word Analogy with Vector Arithmetic / 벡터 산술을 통한 단어 유추
5. **Part 5**: Comparison with Gensim / Gensim과 비교
6. **Summary**: Paper vs. Modern Equivalents / 논문 vs. 현대 동등물

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from typing import Optional

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## Part 1: Vocabulary & Training Data / 어휘 및 학습 데이터 구축

논문에서는 16억 단어의 Google News 코퍼스를 사용했지만, 여기서는 알고리즘의 동작 원리를 이해하기 위해 작은 toy corpus를 사용합니다. 핵심 개념은 동일합니다:
- 텍스트를 토큰화하고 어휘(vocabulary)를 구축
- Context window 내의 (context, target) 쌍을 생성
- One-hot encoding으로 단어를 표현

The paper used a 1.6 billion word Google News corpus, but we use a small toy corpus here to understand how the algorithm works. The core concepts are identical:
- Tokenize text and build vocabulary
- Generate (context, target) pairs within a context window
- Represent words with one-hot encoding

In [ ]:
# Toy corpus with semantic relationships that word2vec can learn
# Designed so that similar words appear in similar contexts
corpus = [
    "the king rules the kingdom with power",
    "the queen rules the kingdom with grace",
    "the king and queen live in the castle",
    "the prince is the son of the king",
    "the princess is the daughter of the queen",
    "the man works in the city",
    "the woman works in the city",
    "the boy plays in the garden",
    "the girl plays in the garden",
    "the king has a crown of gold",
    "the queen has a crown of silver",
    "the man and woman live in the city",
    "the prince and princess live in the castle",
    "the boy is the son of the man",
    "the girl is the daughter of the woman",
    "a king is a man who rules",
    "a queen is a woman who rules",
    "the prince will be the future king",
    "the princess will be the future queen",
    "the dog runs in the garden",
    "the cat sleeps in the castle",
]

def tokenize(corpus: list[str]) -> list[list[str]]:
    """Tokenize corpus into list of word lists."""
    return [sentence.lower().split() for sentence in corpus]

def build_vocab(tokenized_corpus: list[list[str]]) -> tuple[dict, dict]:
    """Build word-to-index and index-to-word mappings."""
    word_counts = Counter(w for sent in tokenized_corpus for w in sent)
    # Sort by frequency (descending) for consistency
    vocab = sorted(word_counts.keys(), key=lambda w: -word_counts[w])
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    return word2idx, idx2word

tokenized = tokenize(corpus)
word2idx, idx2word = build_vocab(tokenized)
V = len(word2idx)  # Vocabulary size

print(f"Vocabulary size (V): {V}")
print(f"Total tokens: {sum(len(s) for s in tokenized)}")
print(f"\nVocabulary (sorted by frequency):")
word_counts = Counter(w for sent in tokenized for w in sent)
for w in list(word2idx.keys())[:15]:
    print(f"  {w:12s} → idx {word2idx[w]:2d}  (count: {word_counts[w]})")

In [ ]:
def generate_cbow_pairs(
    tokenized_corpus: list[list[str]],
    word2idx: dict,
    window_size: int = 2
) -> tuple[np.ndarray, np.ndarray]:
    """Generate (context_indices, target_index) pairs for CBOW.

    For CBOW, context = surrounding words, target = center word.
    The paper uses 4 past + 4 future words (window_size=4).
    We use window_size=2 for our toy corpus.

    Returns:
        contexts: array of shape (num_pairs, 2*window_size), context word indices
        targets: array of shape (num_pairs,), target word indices
    """
    contexts, targets = [], []
    for sentence in tokenized_corpus:
        indices = [word2idx[w] for w in sentence]
        for i in range(window_size, len(indices) - window_size):
            ctx = indices[i - window_size:i] + indices[i + 1:i + window_size + 1]
            contexts.append(ctx)
            targets.append(indices[i])
    return np.array(contexts), np.array(targets)


def generate_skipgram_pairs(
    tokenized_corpus: list[list[str]],
    word2idx: dict,
    window_size: int = 2
) -> tuple[np.ndarray, np.ndarray]:
    """Generate (center_index, context_index) pairs for Skip-gram.

    For Skip-gram, input = center word, output = one surrounding word.
    The paper randomly samples R in [1, C] for each training word.

    Returns:
        centers: array of shape (num_pairs,), center word indices
        contexts: array of shape (num_pairs,), context word indices
    """
    centers, contexts = [], []
    for sentence in tokenized_corpus:
        indices = [word2idx[w] for w in sentence]
        for i in range(len(indices)):
            # Paper: randomly sample R from [1, C]
            R = np.random.randint(1, window_size + 1)
            for j in range(max(0, i - R), min(len(indices), i + R + 1)):
                if j != i:
                    centers.append(indices[i])
                    contexts.append(indices[j])
    return np.array(centers), np.array(contexts)


# Generate training pairs
cbow_contexts, cbow_targets = generate_cbow_pairs(tokenized, word2idx, window_size=2)
sg_centers, sg_contexts = generate_skipgram_pairs(tokenized, word2idx, window_size=2)

print(f"CBOW training pairs: {len(cbow_targets)}")
print(f"Skip-gram training pairs: {len(sg_centers)}")
print(f"\nExample CBOW pair:")
print(f"  Context: {[idx2word[i] for i in cbow_contexts[5]]} → Target: {idx2word[cbow_targets[5]]}")
print(f"\nExample Skip-gram pair:")
print(f"  Center: {idx2word[sg_centers[5]]} → Context: {idx2word[sg_contexts[5]]}")

## Part 2: CBOW from Scratch / CBOW 직접 구현

CBOW 아키텍처는 NNLM에서 hidden layer를 제거하고, projection layer를 공유하며, context 단어 벡터를 **평균**하여 target 단어를 예측합니다. 논문에서는 hierarchical softmax를 사용했지만, 교육 목적으로 여기서는 full softmax를 사용합니다.

The CBOW architecture removes the hidden layer from NNLM, shares the projection layer, and **averages** context word vectors to predict the target word. The paper uses hierarchical softmax, but we use full softmax here for educational purposes.

### Architecture / 아키텍처:
```
Context words → Projection (shared) → Average → Output (softmax)
[w(t-2), w(t-1), w(t+1), w(t+2)] → [v1, v2, v3, v4] → mean(v) → softmax → w(t)
```

### Complexity / 복잡도:
$$Q_{\text{CBOW}} = N \times D + D \times \log_2(V) \quad \text{(with hierarchical softmax)}$$
$$Q_{\text{CBOW}} = N \times D + D \times V \quad \text{(with full softmax, our implementation)}$$

In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


class CBOW:
    """Continuous Bag-of-Words model (Mikolov et al., 2013).

    Architecture: context words → shared projection → average → softmax output
    No hidden layer — this is the key simplification over NNLM.

    Attributes:
        W_in: Input weight matrix (V × D) — the word vectors we want to learn
        W_out: Output weight matrix (D × V) — softmax weights
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        """Initialize with random weights.

        Args:
            vocab_size: Size of vocabulary (V)
            embedding_dim: Dimensionality of word vectors (D)
        """
        self.V = vocab_size
        self.D = embedding_dim
        # Xavier initialization
        scale = np.sqrt(2.0 / (vocab_size + embedding_dim))
        self.W_in = np.random.randn(vocab_size, embedding_dim) * scale
        self.W_out = np.random.randn(embedding_dim, vocab_size) * scale

    def forward(self, context_indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Forward pass: context → average embedding → softmax probabilities.

        Args:
            context_indices: Array of context word indices, shape (N,)

        Returns:
            h: Hidden state (averaged projection), shape (D,)
            y_pred: Predicted probability distribution, shape (V,)
        """
        # Step 1: Look up embeddings for all context words (projection layer)
        # Paper: "all words get projected into the same position"
        embeddings = self.W_in[context_indices]  # (N, D)

        # Step 2: Average the projections (this is the "bag-of-words" — order is lost)
        h = np.mean(embeddings, axis=0)  # (D,)

        # Step 3: Compute output scores and softmax
        scores = h @ self.W_out  # (V,)
        y_pred = softmax(scores)  # (V,)

        return h, y_pred

    def train(
        self,
        contexts: np.ndarray,
        targets: np.ndarray,
        epochs: int = 100,
        lr: float = 0.025,
        print_every: int = 20
    ) -> list[float]:
        """Train CBOW with SGD.

        The paper uses SGD with learning rate starting at 0.025,
        decreasing linearly to 0 at the end of training.

        Args:
            contexts: Context word indices, shape (num_samples, N)
            targets: Target word indices, shape (num_samples,)
            epochs: Number of training epochs
            lr: Initial learning rate
            print_every: Print loss every N epochs

        Returns:
            List of average losses per epoch
        """
        losses = []
        num_samples = len(targets)

        for epoch in range(epochs):
            epoch_loss = 0.0
            # Linear learning rate decay (as in the paper)
            current_lr = lr * (1 - epoch / epochs)
            current_lr = max(current_lr, lr * 0.0001)

            # Shuffle training data
            perm = np.random.permutation(num_samples)

            for idx in perm:
                ctx = contexts[idx]
                target = targets[idx]

                # Forward pass
                h, y_pred = self.forward(ctx)

                # Compute cross-entropy loss
                epoch_loss -= np.log(y_pred[target] + 1e-10)

                # Backward pass
                # Gradient of softmax cross-entropy: y_pred - y_true
                grad_scores = y_pred.copy()  # (V,)
                grad_scores[target] -= 1.0

                # Gradient for W_out: outer product of h and grad_scores
                grad_W_out = np.outer(h, grad_scores)  # (D, V)

                # Gradient for h (averaged embedding)
                grad_h = self.W_out @ grad_scores  # (D,)

                # Gradient for W_in: distribute gradient equally to all context words
                # Because h = mean(embeddings), grad for each embedding = grad_h / N
                grad_embed = grad_h / len(ctx)

                # Update weights
                self.W_out -= current_lr * grad_W_out
                for c in ctx:
                    self.W_in[c] -= current_lr * grad_embed

            avg_loss = epoch_loss / num_samples
            losses.append(avg_loss)
            if (epoch + 1) % print_every == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}, Loss: {avg_loss:.4f}, LR: {current_lr:.5f}")

        return losses

    def get_embedding(self, word: str, word2idx: dict) -> np.ndarray:
        """Get the learned word vector for a word."""
        return self.W_in[word2idx[word]]


# Train CBOW
D = 20  # Embedding dimensionality (paper uses 300-1000, we use 20 for toy data)
print(f"Training CBOW: V={V}, D={D}")
print(f"Complexity per sample: Q = N×D + D×V = {4*D} + {D*V} = {4*D + D*V}")
print()

cbow_model = CBOW(V, D)
cbow_losses = cbow_model.train(cbow_contexts, cbow_targets, epochs=200, lr=0.05, print_every=40)

## Part 3: Skip-gram from Scratch / Skip-gram 직접 구현

Skip-gram은 CBOW의 **역방향**입니다: 중심 단어가 주어지면 주변 단어를 예측합니다. 논문에서 Skip-gram은 의미적 관계(semantic relationships)에서 더 우수한 성능을 보였습니다.

Skip-gram is the **reverse** of CBOW: given the center word, predict surrounding words. In the paper, Skip-gram showed superior performance on semantic relationships.

### Architecture / 아키텍처:
```
Center word → Projection → Output (softmax) → Context words
w(t) → v_w(t) → softmax → [w(t-2), w(t-1), w(t+1), w(t+2)]
```

### Complexity / 복잡도:
$$Q_{\text{Skip-gram}} = C \times (D + D \times \log_2(V))$$

In [ ]:
class SkipGram:
    """Skip-gram model (Mikolov et al., 2013).

    Architecture: center word → projection → softmax output → context words
    Predicts each context word independently given the center word.

    Attributes:
        W_in: Input weight matrix (V × D) — the word vectors we want to learn
        W_out: Output weight matrix (D × V) — softmax weights
    """

    def __init__(self, vocab_size: int, embedding_dim: int):
        self.V = vocab_size
        self.D = embedding_dim
        scale = np.sqrt(2.0 / (vocab_size + embedding_dim))
        self.W_in = np.random.randn(vocab_size, embedding_dim) * scale
        self.W_out = np.random.randn(embedding_dim, vocab_size) * scale

    def forward(self, center_idx: int) -> tuple[np.ndarray, np.ndarray]:
        """Forward pass: center word → embedding → softmax probabilities.

        Args:
            center_idx: Index of the center word

        Returns:
            h: Center word embedding, shape (D,)
            y_pred: Predicted probability distribution, shape (V,)
        """
        # Step 1: Look up embedding for center word
        h = self.W_in[center_idx]  # (D,)

        # Step 2: Compute output scores and softmax
        scores = h @ self.W_out  # (V,)
        y_pred = softmax(scores)  # (V,)

        return h, y_pred

    def train(
        self,
        centers: np.ndarray,
        contexts: np.ndarray,
        epochs: int = 100,
        lr: float = 0.025,
        print_every: int = 20
    ) -> list[float]:
        """Train Skip-gram with SGD.

        Each (center, context) pair is one training example.
        The paper samples R ∈ [1, C] for each training word.

        Args:
            centers: Center word indices, shape (num_samples,)
            contexts: Context word indices, shape (num_samples,)
            epochs: Number of training epochs
            lr: Initial learning rate
            print_every: Print loss every N epochs

        Returns:
            List of average losses per epoch
        """
        losses = []
        num_samples = len(centers)

        for epoch in range(epochs):
            epoch_loss = 0.0
            current_lr = lr * (1 - epoch / epochs)
            current_lr = max(current_lr, lr * 0.0001)

            perm = np.random.permutation(num_samples)

            for idx in perm:
                center = centers[idx]
                context = contexts[idx]

                # Forward pass
                h, y_pred = self.forward(center)

                # Loss
                epoch_loss -= np.log(y_pred[context] + 1e-10)

                # Backward pass
                grad_scores = y_pred.copy()
                grad_scores[context] -= 1.0

                # Gradient for W_out
                grad_W_out = np.outer(h, grad_scores)

                # Gradient for W_in (only the center word's embedding)
                grad_h = self.W_out @ grad_scores

                # Update weights
                self.W_out -= current_lr * grad_W_out
                self.W_in[center] -= current_lr * grad_h

            avg_loss = epoch_loss / num_samples
            losses.append(avg_loss)
            if (epoch + 1) % print_every == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}, Loss: {avg_loss:.4f}, LR: {current_lr:.5f}")

        return losses

    def get_embedding(self, word: str, word2idx: dict) -> np.ndarray:
        """Get the learned word vector for a word."""
        return self.W_in[word2idx[word]]


# Train Skip-gram
print(f"Training Skip-gram: V={V}, D={D}")
print(f"Complexity per sample: Q = D + D×V = {D} + {D*V} = {D + D*V}")
print()

sg_model = SkipGram(V, D)
sg_losses = sg_model.train(sg_centers, sg_contexts, epochs=200, lr=0.05, print_every=40)

In [ ]:
# Plot training losses for both models
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(cbow_losses, label='CBOW', alpha=0.8)
ax.plot(sg_losses, label='Skip-gram', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Average Cross-Entropy Loss')
ax.set_title('Training Loss: CBOW vs Skip-gram / 학습 손실 비교')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Word Analogy with Vector Arithmetic / 벡터 산술을 통한 단어 유추

논문의 가장 유명한 결과는 벡터 산술로 단어 관계를 풀 수 있다는 것입니다:

The paper's most famous result is solving word relationships through vector arithmetic:

$$X = \text{vector}(b) - \text{vector}(a) + \text{vector}(c)$$
$$\text{answer} = \arg\max_{w} \cos(X, \text{vector}(w))$$

예를 들어 / For example: "king is to queen as man is to ?" → **woman**

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)


def find_most_similar(
    word: str,
    embeddings: np.ndarray,
    word2idx: dict,
    idx2word: dict,
    top_k: int = 5
) -> list[tuple[str, float]]:
    """Find the top-k most similar words by cosine similarity."""
    if word not in word2idx:
        return []
    vec = embeddings[word2idx[word]]
    similarities = []
    for i in range(len(idx2word)):
        if idx2word[i] != word:
            sim = cosine_similarity(vec, embeddings[i])
            similarities.append((idx2word[i], sim))
    similarities.sort(key=lambda x: -x[1])
    return similarities[:top_k]


def word_analogy(
    a: str, b: str, c: str,
    embeddings: np.ndarray,
    word2idx: dict,
    idx2word: dict,
    top_k: int = 3
) -> list[tuple[str, float]]:
    """Solve analogy: a is to b as c is to ?

    Computes: vector(b) - vector(a) + vector(c), then finds closest word.
    Excludes a, b, c from results (as in the paper).
    """
    exclude = {a, b, c}
    vec = embeddings[word2idx[b]] - embeddings[word2idx[a]] + embeddings[word2idx[c]]
    similarities = []
    for i in range(len(idx2word)):
        if idx2word[i] not in exclude:
            sim = cosine_similarity(vec, embeddings[i])
            similarities.append((idx2word[i], sim))
    similarities.sort(key=lambda x: -x[1])
    return similarities[:top_k]


# Test word similarities
print("=" * 60)
print("Word Similarities (Skip-gram) / 단어 유사도 (Skip-gram)")
print("=" * 60)
for word in ["king", "queen", "man", "woman", "boy", "girl", "prince"]:
    similar = find_most_similar(word, sg_model.W_in, word2idx, idx2word, top_k=5)
    sim_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:10s} → {sim_str}")

print()
print("=" * 60)
print("Word Similarities (CBOW) / 단어 유사도 (CBOW)")
print("=" * 60)
for word in ["king", "queen", "man", "woman", "boy", "girl", "prince"]:
    similar = find_most_similar(word, cbow_model.W_in, word2idx, idx2word, top_k=5)
    sim_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:10s} → {sim_str}")

In [ ]:
# Word analogy tests
# Paper's famous example: king - man + woman ≈ queen
print("=" * 60)
print("Word Analogies (Skip-gram) / 단어 유추 (Skip-gram)")
print("=" * 60)

analogies = [
    ("king", "queen", "man", "woman"),      # king:queen = man:?
    ("king", "queen", "prince", "princess"), # king:queen = prince:?
    ("man", "woman", "boy", "girl"),         # man:woman = boy:?
    ("king", "prince", "queen", "princess"), # king:prince = queen:?
]

for a, b, c, expected in analogies:
    results = word_analogy(a, b, c, sg_model.W_in, word2idx, idx2word, top_k=3)
    top_word = results[0][0] if results else "?"
    mark = "✓" if top_word == expected else "✗"
    result_str = ", ".join([f"{w}({s:.2f})" for w, s in results])
    print(f"  {a}:{b} = {c}:? → {result_str}  (expected: {expected}) {mark}")

print()
print("=" * 60)
print("Word Analogies (CBOW) / 단어 유추 (CBOW)")
print("=" * 60)

for a, b, c, expected in analogies:
    results = word_analogy(a, b, c, cbow_model.W_in, word2idx, idx2word, top_k=3)
    top_word = results[0][0] if results else "?"
    mark = "✓" if top_word == expected else "✗"
    result_str = ", ".join([f"{w}({s:.2f})" for w, s in results])
    print(f"  {a}:{b} = {c}:? → {result_str}  (expected: {expected}) {mark}")

In [ ]:
# Visualize word vectors in 2D using PCA
from sklearn.decomposition import PCA

def plot_embeddings(
    embeddings: np.ndarray,
    idx2word: dict,
    title: str,
    highlight_words: Optional[list[str]] = None,
    ax: Optional[plt.Axes] = None
):
    """Plot word embeddings in 2D using PCA."""
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeddings)

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))

    # Plot all words in gray
    ax.scatter(coords[:, 0], coords[:, 1], c='lightgray', s=30, alpha=0.5)

    # Highlight specific words
    highlight = highlight_words or [
        "king", "queen", "prince", "princess",
        "man", "woman", "boy", "girl",
        "castle", "city", "garden"
    ]
    colors = {
        "king": "royalblue", "queen": "crimson",
        "prince": "cornflowerblue", "princess": "lightcoral",
        "man": "darkblue", "woman": "darkred",
        "boy": "steelblue", "girl": "indianred",
        "castle": "goldenrod", "city": "forestgreen", "garden": "olive",
    }

    for word in highlight:
        if word in word2idx:
            i = word2idx[word]
            c = colors.get(word, "black")
            ax.scatter(coords[i, 0], coords[i, 1], c=c, s=120, zorder=5, edgecolors='black')
            ax.annotate(word, (coords[i, 0], coords[i, 1]),
                       fontsize=11, fontweight='bold',
                       xytext=(5, 5), textcoords='offset points')

    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    var_explained = pca.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({var_explained[0]:.1%})')
    ax.set_ylabel(f'PC2 ({var_explained[1]:.1%})')


fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_embeddings(cbow_model.W_in, idx2word, "CBOW Word Vectors (PCA)", ax=axes[0])
plot_embeddings(sg_model.W_in, idx2word, "Skip-gram Word Vectors (PCA)", ax=axes[1])
plt.suptitle("Learned Word Embeddings / 학습된 단어 임베딩", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 5: Comparison with Gensim / Gensim과 비교

`gensim`은 Word2Vec의 가장 널리 사용되는 현대적 구현입니다. 논문의 원본 C++ 코드(`word2vec`)를 Python으로 최적화하여 구현한 것으로, negative sampling, subsampling 등 후속 논문의 개선 사항도 포함합니다.

`gensim` is the most widely used modern implementation of Word2Vec. It's an optimized Python implementation of the original C++ code (`word2vec`), including improvements from follow-up papers like negative sampling and subsampling.

우리의 toy corpus에서도 gensim이 더 나은 결과를 보여주는지 비교합니다.

We compare whether gensim shows better results even on our toy corpus.

In [ ]:
try:
    from gensim.models import Word2Vec

    # Train gensim Word2Vec with Skip-gram (sg=1) and CBOW (sg=0)
    # min_count=1 to keep all words in our small corpus
    gensim_sg = Word2Vec(
        sentences=tokenized,
        vector_size=D,
        window=2,
        min_count=1,
        sg=1,           # Skip-gram
        epochs=200,
        seed=42,
        workers=1,
    )

    gensim_cbow = Word2Vec(
        sentences=tokenized,
        vector_size=D,
        window=2,
        min_count=1,
        sg=0,           # CBOW
        epochs=200,
        seed=42,
        workers=1,
    )

    print("Gensim Skip-gram — Most similar words:")
    for word in ["king", "queen", "man", "woman"]:
        similar = gensim_sg.wv.most_similar(word, topn=5)
        sim_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
        print(f"  {word:10s} → {sim_str}")

    print("\nGensim CBOW — Most similar words:")
    for word in ["king", "queen", "man", "woman"]:
        similar = gensim_cbow.wv.most_similar(word, topn=5)
        sim_str = ", ".join([f"{w}({s:.2f})" for w, s in similar])
        print(f"  {word:10s} → {sim_str}")

    print("\nGensim Skip-gram — Analogies:")
    for a, b, c, expected in analogies:
        try:
            result = gensim_sg.wv.most_similar(positive=[b, c], negative=[a], topn=3)
            top_word = result[0][0]
            mark = "✓" if top_word == expected else "✗"
            result_str = ", ".join([f"{w}({s:.2f})" for w, s in result])
            print(f"  {a}:{b} = {c}:? → {result_str}  (expected: {expected}) {mark}")
        except KeyError as e:
            print(f"  {a}:{b} = {c}:? → KeyError: {e}")

except ImportError:
    print("gensim is not installed. Install with: pip install gensim")
    print("Skipping gensim comparison.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Word embedding | `W_in` matrix (V×D), learned via prediction task | `nn.Embedding` in PyTorch, pre-trained models (GloVe, FastText) |
| CBOW architecture | Context → average → predict center word | Still available in gensim; less popular than Skip-gram |
| Skip-gram architecture | Center word → predict context words | Default in most Word2Vec implementations; basis for FastText |
| Hierarchical softmax | Huffman tree-based output layer, $O(\log V)$ | Replaced by negative sampling in practice (simpler, comparable quality) |
| Full softmax | $O(V)$ output layer | Used in small-vocab settings; replaced by sampled softmax for large vocabs |
| Linear LR decay | `lr *= (1 - epoch/total_epochs)` | Common in modern training; also cosine annealing, warmup schedules |
| Vector arithmetic | `vec(b) - vec(a) + vec(c)` for analogies | Still used for evaluation; modern models use contextual embeddings (BERT) |
| Word similarity eval | Cosine distance on static vectors | Contextual similarity (different vectors per context in ELMo/BERT) |

### 논문의 구현에서 생략된 것 / What we simplified from the paper:
1. **Hierarchical softmax** → Full softmax 사용 (교육 목적 / for educational purposes)
2. **Negative sampling** → 후속 논문(2013b)의 내용으로, 여기서는 미구현 / From follow-up paper, not implemented here
3. **Subsampling of frequent words** → 후속 논문의 기법 / Technique from follow-up paper
4. **대규모 데이터** → Toy corpus 사용. 실제 성능은 수십억 단어 데이터에서 발현 / Used toy corpus; real performance emerges with billions of words
5. **DistBelief 병렬 학습** → 단일 프로세스 / Single process training